In [1]:
import sys; sys.path.insert(0, "..")
from fantasy.yahoo_client import get_my_roster, get_current_matchup
from fantasy.nba_stats import get_games_this_week, batch_get_player_stats
from fantasy.analysis import project_player
from fantasy.llm import build_start_sit_prompt, ask_gemini
import pandas as pd

In [2]:
matchup = get_current_matchup()
week_start, week_end = matchup["week_start"], matchup["week_end"]
my_roster = get_my_roster()

with_stats = batch_get_player_stats(my_roster)
players = [{**p, "games_remaining": get_games_this_week(p["team_abbr"], week_start, week_end)}
           for p in with_stats]

# Sort by projected PTS descending
def projected_pts(p):
    if p["stats"] is None:
        return 0
    proj = project_player(p["stats"], p["games_remaining"], p.get("status", ""))
    return proj.get("PTS", 0)

players.sort(key=projected_pts, reverse=True)

[2026-03-23 22:57:03,102 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-03-23 22:57:03,106 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 1372.5197026729584
[2026-03-23 22:57:03,107 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-23 22:57:03,111 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 1372.5243754386902
[2026-03-23 22:57:03,112 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-23 22:57:04,137 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-03-23 22:57:04,142 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 1373.5555806159973
[2026-03-23 22:57:04,143 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-23 22:57:04,147 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 1373.5606396198273
[2026-03-23 22:57:04,148 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID


In [3]:
import sys; sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("/mnt/c/Users/louis/Documents/dev/nba_fantasy/.env", override=True)

from fantasy.yahoo_client import _get
import os, json

TEAM_KEY = os.environ.get("YAHOO_TEAM_KEY")
data = _get(f"/team/{TEAM_KEY}/roster")

# Look at raw structure of first player
players_raw = data["team"][1]["roster"]["0"]["players"]
print(json.dumps(players_raw["0"], indent=2))

[2026-03-23 23:00:29,951 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-03-23 23:00:29,955 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 1579.3682088851929
[2026-03-23 23:00:29,955 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-23 23:00:29,966 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 1579.3788847923279
[2026-03-23 23:00:29,966 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID


{
  "player": [
    [
      {
        "player_key": "466.p.5842"
      },
      {
        "player_id": "5842"
      },
      {
        "name": {
          "full": "Derrick White",
          "first": "Derrick",
          "last": "White",
          "ascii_first": "Derrick",
          "ascii_last": "White"
        }
      },
      {
        "url": "https://sports.yahoo.com/nba/players/5842"
      },
      {
        "editorial_player_key": "nba.p.5842"
      },
      {
        "editorial_team_key": "nba.t.2"
      },
      {
        "editorial_team_full_name": "Boston Celtics"
      },
      {
        "editorial_team_abbr": "BOS"
      },
      {
        "editorial_team_url": "https://sports.yahoo.com/nba/teams/boston/"
      },
      {
        "is_keeper": {
          "status": false,
          "cost": false,
          "kept": false
        }
      },
      {
        "uniform_number": "9"
      },
      {
        "display_position": "PG,SG"
      },
      {
        "headshot": {
         

In [4]:
rows = []
for p in players:
    rows.append({
        "Player": p["name"],
        "Pos": p["position"],
        "Status": p["status"] or "Active",
        "Games Left": p["games_remaining"],
        "Proj PTS": round(p["projected"].get("PTS", 0), 1),
        "Proj REB": round(p["projected"].get("REB", 0), 1),
        "Proj AST": round(p["projected"].get("AST", 0), 1),
    })
print(pd.DataFrame(rows).to_string(index=False))

            Player      Pos Status  Games Left  Proj PTS  Proj REB  Proj AST
    Alperen Sengun     PF,C Active           1     565.1     243.4     166.3
     Tobias Harris       PF Active           1     338.0     139.5      64.2
 Immanuel Quickley    PG,SG    GTD           1     251.3      56.3      89.7
  Taylor Hendricks  SF,PF,C Active           1     233.2      99.3      29.6
     Derrick White    PG,SG Active           0       0.0       0.0       0.0
      Cooper Flagg PG,SG,SF Active           1       0.0       0.0       0.0
       Ayo Dosunmu PG,SG,SF Active           0       0.0       0.0       0.0
  Precious Achiuwa     PF,C    GTD           0       0.0       0.0       0.0
      Nikola Jokić        C Active           0       0.0       0.0       0.0
   Donovan Clingan        C Active           1       0.0       0.0       0.0
  Donovan Mitchell    PG,SG Active           0       0.0       0.0       0.0
     Miles Bridges    SF,PF Active           0       0.0       0.0       0.0

In [5]:
prompt = build_start_sit_prompt(players)
advice = ask_gemini(prompt)
print("\n=== TODAY'S LINEUP CARD ===\n")
print(advice)


=== TODAY'S LINEUP CARD ===

Okay, here are my start/sit recommendations based on your projected point totals and player status, focusing on maximizing your single remaining game:

*   **Alperen Sengun: START** - He's clearly your best option with a massive projected point total.
*   **Tobias Harris: START** - Harris offers a strong projected point total and should be in your lineup.
*   **Immanuel Quickley: SIT** - GTD status makes him too risky, as you can't afford a 0 if he doesn't play.
*   **Taylor Hendricks: START** - With decent projected points, Hendricks is a must-start with limited options.
*   **Derrick White: SIT** - He's not playing, so obviously sit him.
*   **Cooper Flagg: SIT** - Zero projected points means he's a benchwarmer.
*   **Ayo Dosunmu: SIT** - He's not playing, so he shouldn't be in your lineup.
*   **Precious Achiuwa: SIT** - The GTD status combined with zero projected points makes him a huge risk.
*   **Nikola Jokić: SIT** - He's not playing, keep him on th